# 5.2 Attention-Based ECG–PCG Fusion

This experiment evaluates attention-based multimodal fusion using the same
five-fold subject-level cross-validation protocol as the ECG-only, PCG-only,
and simple concatenation fusion experiments.

For every fold, the corresponding pretrained ECG and PCG encoders are reused
and frozen. Their patient-level embeddings are projected into a common latent
space and treated as two modality tokens. Multi-head self-attention is then
used to learn interactions between ECG and PCG representations before final
HFrEF classification.

Primary comparison:
- ECG only
- PCG only
- Simple concatenation fusion
- Attention fusion


In [4]:
from pathlib import Path
import sys
import json
import torch
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: C:\Users\Dell\Documents\Deep_Learning\ecg-pcg-fusion-hfref
PyTorch: 2.13.0+cpu
CUDA available: False


In [5]:
import importlib

import src.fusion_model as fusion_model
import src.fusion_five_fold as fusion_five_fold

importlib.reload(fusion_model)
importlib.reload(fusion_five_fold)

from src.data_loader import ProcessedCardioDataset
from torch.utils.data import DataLoader

print("Fusion modules reloaded successfully.")

Fusion modules reloaded successfully.


In [6]:
import inspect

print(inspect.signature(fusion_model.build_fusion_model))
print(inspect.signature(fusion_five_fold.train_fusion_five_fold))

(ecg_checkpoint_path, ecg_model_config, pcg_checkpoint_path, pcg_model_config, architecture='concat', hidden_dim=128, dropout=0.3, freeze_encoders=True, attention_dim=128, num_heads=4, device='cpu')
(project_dir, architecture='concat', encoder_seed=42, fusion_seed=42, batch_size=16, learning_rate=0.001, max_epochs=50, patience=10, freeze_encoders=True, hidden_dim=128, dropout=0.3, attention_dim=128, num_heads=4)


In [7]:
for modality in ["ecg", "pcg"]:

    root = (
        PROJECT_ROOT
        / "models"
        / "five_fold_cross_validation"
        / f"{modality}_seed42"
    )

    print(f"\n{modality.upper()}")

    for fold in range(5):

        checkpoint = (
            root
            / f"fold_{fold}"
            / "best_model.pt"
        )

        settings = (
            root
            / f"fold_{fold}"
            / "settings.json"
        )

        print(
            f"Fold {fold}:",
            "checkpoint =", checkpoint.is_file(),
            "| settings =", settings.is_file(),
        )


ECG
Fold 0: checkpoint = True | settings = True
Fold 1: checkpoint = True | settings = True
Fold 2: checkpoint = True | settings = True
Fold 3: checkpoint = True | settings = True
Fold 4: checkpoint = True | settings = True

PCG
Fold 0: checkpoint = True | settings = True
Fold 1: checkpoint = True | settings = True
Fold 2: checkpoint = True | settings = True
Fold 3: checkpoint = True | settings = True
Fold 4: checkpoint = True | settings = True


In [8]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ecg_fold0 = (
    PROJECT_ROOT
    / "models"
    / "five_fold_cross_validation"
    / "ecg_seed42"
    / "fold_0"
)

pcg_fold0 = (
    PROJECT_ROOT
    / "models"
    / "five_fold_cross_validation"
    / "pcg_seed42"
    / "fold_0"
)

ecg_config = json.loads(
    (ecg_fold0 / "settings.json").read_text(
        encoding="utf-8"
    )
)["model_config"]

pcg_config = json.loads(
    (pcg_fold0 / "settings.json").read_text(
        encoding="utf-8"
    )
)["model_config"]

attention_model = fusion_model.build_fusion_model(
    ecg_checkpoint_path=ecg_fold0 / "best_model.pt",
    ecg_model_config=ecg_config,
    pcg_checkpoint_path=pcg_fold0 / "best_model.pt",
    pcg_model_config=pcg_config,
    architecture="attention",
    attention_dim=128,
    num_heads=4,
    hidden_dim=128,
    dropout=0.3,
    freeze_encoders=True,
    device=device,
)

print(type(attention_model).__name__)

AttentionFusionModel


In [9]:
ecg_trainable = sum(
    p.numel()
    for p in attention_model.ecg_encoder.parameters()
    if p.requires_grad
)

pcg_trainable = sum(
    p.numel()
    for p in attention_model.pcg_encoder.parameters()
    if p.requires_grad
)

attention_trainable = sum(
    p.numel()
    for p in attention_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in attention_model.parameters()
)

print("Trainable ECG encoder parameters:", ecg_trainable)
print("Trainable PCG encoder parameters:", pcg_trainable)
print("Trainable fusion parameters:", attention_trainable)
print("Total model parameters:", total_params)

Trainable ECG encoder parameters: 0
Trainable PCG encoder parameters: 0
Trainable fusion parameters: 165633
Total model parameters: 1212547


In [10]:
from src.five_fold_validation import _load_metadata

metadata, development_dir = (
    _load_metadata(
        PROJECT_ROOT,
        "development"
    )
)

fold0_df = metadata[
    metadata["Fold"] == 0
].reset_index(drop=True)

fold0_dataset = ProcessedCardioDataset(
    fold0_df,
    development_dir,
    modality="both",
)

fold0_loader = DataLoader(
    fold0_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

ecg, pcg, labels = next(
    iter(fold0_loader)
)

print("ECG:", ecg.shape)
print("PCG:", pcg.shape)
print("Labels:", labels.shape)

ECG: torch.Size([2, 4, 15000])
PCG: torch.Size([2, 4, 120000])
Labels: torch.Size([2])


In [11]:
attention_model.eval()

with torch.no_grad():

    logits, attention_weights = (
        attention_model.forward_with_attention(
            ecg.to(device),
            pcg.to(device),
        )
    )

print("Logits shape:")
print(logits.shape)

print("\nAttention weights shape:")
print(attention_weights.shape)

print("\nAttention matrices:")
print(attention_weights.cpu())

Logits shape:
torch.Size([2, 1])

Attention weights shape:
torch.Size([2, 2, 2])

Attention matrices:
tensor([[[0.4370, 0.5630],
         [0.4871, 0.5129]],

        [[0.3917, 0.6083],
         [0.4860, 0.5140]]])


In [12]:
attention_result_dir = (
    fusion_five_fold.train_fusion_five_fold(
        project_dir=PROJECT_ROOT,
        architecture="attention",
        encoder_seed=42,
        fusion_seed=42,
        batch_size=16,
        learning_rate=1e-3,
        max_epochs=50,
        patience=10,
        freeze_encoders=True,
        hidden_dim=128,
        dropout=0.3,
        attention_dim=128,
        num_heads=4,
    )
)

print(
    "Attention results saved to:",
    attention_result_dir
)

FUSION fold 0 | epoch 1 | train loss 0.0044 | validation AUROC 0.9413 | average precision 0.6056
FUSION fold 0 | epoch 2 | train loss 0.0000 | validation AUROC 0.9403 | average precision 0.6043
FUSION fold 0 | epoch 3 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 4 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 5 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 6 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 7 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 8 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 9 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6096
FUSION fold 0 | epoch 10 | train loss 0.0000 | validation AUROC 0.9413 | average precision 0.6142
FUSION fold 0 | epoch 11 | tr

In [13]:
attention_summary_path = (
    PROJECT_ROOT
    / "analysis_outputs"
    / "five_fold_cross_validation"
    / "fusion_attention_seed42"
    / "development_summary.csv"
)

attention_summary = pd.read_csv(
    attention_summary_path
)

display(
    attention_summary.round(4)
)

,modality,architecture,seed,cutoff_rule,development_cutoff,cutoff_selection_specificity,patients,hfref_patients,threshold,auprc,...,balanced_accuracy,precision,f1,brier,log_loss,calibration_error,tn,fp,fn,tp
0,fusion,attention,42,maximum_development_f1,0.7016,NaN,500,60,0.7016,0.4962,...,0.7875,0.5417,0.5909,0.1126,0.3722,0.1273,407,33,21,39
1,fusion,attention,42,target_90pct_development_specificity,0.6291,0.9,500,60,0.6291,0.4962,...,0.7833,0.4762,0.5556,0.1126,0.3722,0.1273,396,44,20,40


In [14]:
concat_path = (
    PROJECT_ROOT
    / "analysis_outputs"
    / "five_fold_cross_validation"
    / "fusion_concat_seed42"
    / "development_summary.csv"
)

attention_path = (
    PROJECT_ROOT
    / "analysis_outputs"
    / "five_fold_cross_validation"
    / "fusion_attention_seed42"
    / "development_summary.csv"
)

concat_summary = pd.read_csv(concat_path)
attention_summary = pd.read_csv(attention_path)

concat_summary["model"] = "Simple Fusion"
attention_summary["model"] = "Attention Fusion"

fusion_comparison = pd.concat(
    [
        concat_summary,
        attention_summary,
    ],
    ignore_index=True,
)

display(
    fusion_comparison[
        [
            "model",
            "cutoff_rule",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "precision",
            "f1",
            "brier",
            "log_loss",
        ]
    ].round(4)
)

,model,cutoff_rule,auroc,auprc,balanced_accuracy,precision,f1,brier,log_loss
0,Simple Fusion,maximum_development_f1,0.9034,0.5735,0.7894,0.6129,0.6230,0.0825,0.2832
1,Simple Fusion,target_90pct_development_specificity,0.9034,0.5735,0.8167,0.5000,0.5946,0.0825,0.2832
2,Attention Fusion,maximum_development_f1,0.8357,0.4962,0.7875,0.5417,0.5909,0.1126,0.3722
3,Attention Fusion,target_90pct_development_specificity,0.8357,0.4962,0.7833,0.4762,0.5556,0.1126,0.3722


In [15]:
base = (
    PROJECT_ROOT
    / "analysis_outputs"
    / "five_fold_cross_validation"
)

paths = {
    "ECG only": base / "ecg_seed42" / "development_summary.csv",
    "PCG only": base / "pcg_seed42" / "development_summary.csv",
    "Simple Fusion": base / "fusion_concat_seed42" / "development_summary.csv",
    "Attention Fusion": base / "fusion_attention_seed42" / "development_summary.csv",
}

frames = []

for model_name, path in paths.items():

    frame = pd.read_csv(path)
    frame["model"] = model_name

    frames.append(frame)

ablation = pd.concat(
    frames,
    ignore_index=True,
)

ablation = ablation[
    ablation["cutoff_rule"]
    == "maximum_development_f1"
]

display(
    ablation[
        [
            "model",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "precision",
            "f1",
            "brier",
            "log_loss",
        ]
    ]
    .sort_values(
        "auroc",
        ascending=False
    )
    .round(4)
)

,model,auroc,auprc,balanced_accuracy,precision,f1,brier,log_loss
4,Simple Fusion,0.9034,0.5735,0.7894,0.6129,0.6230,0.0825,0.2832
0,ECG only,0.8667,0.5217,0.8307,0.5357,0.6250,0.1156,0.3773
6,Attention Fusion,0.8357,0.4962,0.7875,0.5417,0.5909,0.1126,0.3722
2,PCG only,0.7224,0.2751,0.7042,0.2217,0.3456,0.2174,0.6257


In [16]:
base = (
    PROJECT_ROOT
    / "analysis_outputs"
    / "five_fold_cross_validation"
)

simple_folds = pd.read_csv(
    base
    / "fusion_concat_seed42"
    / "fold_results.csv"
)

attention_folds = pd.read_csv(
    base
    / "fusion_attention_seed42"
    / "fold_results.csv"
)

simple_folds["model"] = "Simple Fusion"
attention_folds["model"] = "Attention Fusion"

fold_comparison = pd.concat(
    [simple_folds, attention_folds],
    ignore_index=True
)

display(
    fold_comparison[
        fold_comparison["cutoff_rule"]
        == "maximum_development_f1"
    ][
        [
            "model",
            "validation_fold",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "f1",
        ]
    ].round(4)
)

,model,validation_fold,auroc,auprc,balanced_accuracy,f1
0,Simple Fusion,0,0.9422,0.6290,0.8523,0.7200
1,Simple Fusion,1,0.9233,0.6273,0.8466,0.6923
2,Simple Fusion,2,0.8769,0.5581,0.6553,0.4444
3,Simple Fusion,3,0.9489,0.7098,0.8655,0.6452
4,Simple Fusion,4,0.9527,0.6856,0.7273,0.5455
10,Attention Fusion,0,0.9413,0.6056,0.6496,0.4211
11,Attention Fusion,1,0.9214,0.7298,0.8655,0.6452
12,Attention Fusion,2,0.7860,0.5300,0.6383,0.3810
13,Attention Fusion,3,0.9527,0.6981,0.9129,0.7097
14,Attention Fusion,4,0.9489,0.6512,0.8712,0.6667


## Conclusion

Attention-based fusion did not outperform simple feature concatenation.

Across five-fold out-of-fold development predictions, simple ECG–PCG
concatenation achieved the strongest discrimination (AUROC 0.9034,
AUPRC 0.5735), compared with attention fusion (AUROC 0.8357,
AUPRC 0.4962), ECG-only (AUROC 0.8667, AUPRC 0.5217), and PCG-only
(AUROC 0.7224, AUPRC 0.2751).

These results suggest that ECG and PCG provide complementary information,
but the added complexity of the attention mechanism did not improve
generalisation in this development cohort. Simple concatenation was
therefore selected as the preferred multimodal fusion architecture for
final evaluation.